# 📝 Assignment: Modeling & Evaluation  

## 🚀 Modelling  
In this section, you will build multiple machine learning models and compare their performance.  

### Tasks:  
1. **Train a Logistic Regression model** and validate it using a **classification report**.  
2. **Train a Decision Tree model** and validate it using a **classification report**.  
3. **Train a Random Forest model** and validate it using a **classification report**.  
4. **Compare the models** using a **confusion matrix** and **classification report** to determine the best-performing model.  
5. **Perform hyperparameter tuning** for each model to improve performance. Use `cross_val_score()` on `X_scaled` and `y` to validate results.  

## 📌 Conclusion  
- Identify the **best-performing model** based on evaluation metrics.  
- Specify the **best hyperparameters** used for the top model.  

## 📤 Submission Instructions  
1. **Create a prediction file** using the test dataset with your best-performing model and submit it to **Kaggle**.  
2. **Upload your complete notebook** (including training, evaluation, and predictions) to your **team's GitHub repository**.  

Happy coding! 🚀  


In [1]:
# Import libraries here
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.compose import ColumnTransformer
from category_encoders import OrdinalEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
filepath = "/kaggle/input/spark-2025-heart-failure-prediction/train.csv"

df = pd.read_csv(filepath)
print(df.shape)
df.head()

(56000, 13)
(14000, 12)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,83327,18995,2,162,83.0,120,80,1,1,0,0,0,1
1,86196,17319,1,158,64.0,120,80,1,1,0,0,1,0
2,59158,19017,1,165,95.0,160,100,2,1,0,0,1,1
3,16399,20388,1,164,83.0,150,100,1,1,0,0,1,1
4,29470,18236,1,156,52.0,100,67,1,1,0,0,0,0


## Data Preparation

In [3]:
# Converting the `age` column from days to years
df["age"] = (df["age"] / 365).astype("int")
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,83327,52,2,162,83.0,120,80,1,1,0,0,0,1
1,86196,47,1,158,64.0,120,80,1,1,0,0,1,0
2,59158,52,1,165,95.0,160,100,2,1,0,0,1,1
3,16399,55,1,164,83.0,150,100,1,1,0,0,1,1
4,29470,49,1,156,52.0,100,67,1,1,0,0,0,0


In [4]:
# Searching for missing values
df.isna().sum()

id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64

**Handling the categorical columns**

In [5]:
df.head(3)

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,83327,52,2,162,83.0,120,80,1,1,0,0,0,1
1,86196,47,1,158,64.0,120,80,1,1,0,0,1,0
2,59158,52,1,165,95.0,160,100,2,1,0,0,1,1


In [6]:
# Creating nominal categories
cols_to_change = ["gender","smoke", "alco", "active", "cardio"]

for cols in cols_to_change:
    df[cols] = pd.Categorical(df[cols], ordered=False)

In [7]:
# Creating ordinal categories
df["cholesterol"] = pd.Categorical(df["cholesterol"], categories = [1,2,3], ordered=True)
df["gluc"] = pd.Categorical(df["gluc"], categories = [1,2,3], ordered=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56000 entries, 0 to 55999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   id           56000 non-null  int64   
 1   age          56000 non-null  int64   
 2   gender       56000 non-null  category
 3   height       56000 non-null  int64   
 4   weight       56000 non-null  float64 
 5   ap_hi        56000 non-null  int64   
 6   ap_lo        56000 non-null  int64   
 7   cholesterol  56000 non-null  category
 8   gluc         56000 non-null  category
 9   smoke        56000 non-null  category
 10  alco         56000 non-null  category
 11  active       56000 non-null  category
 12  cardio       56000 non-null  category
dtypes: category(7), float64(1), int64(5)
memory usage: 2.9 MB


**Checking for class imbalance**

In [9]:
df["cardio"].value_counts(normalize=True)

cardio
0    0.500304
1    0.499696
Name: proportion, dtype: float64

## Feature Engineering

In [10]:
# Creating `BMI` column from `weight` and `height`
df["bmi"] = (df["weight"] / (df["height"]/100)**2)

In [11]:
# Creating Mean Arterial Blood Pressure `mabp` from `ap_hi` and `ap_lo`
df["mabp"] = (df["ap_lo"] + ((1/3) * (df["ap_hi"] - df["ap_lo"])))

In [12]:
df.head(6)

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,bmi,mabp
0,83327,52,2,162,83.0,120,80,1,1,0,0,0,1,31.626276,93.333333
1,86196,47,1,158,64.0,120,80,1,1,0,0,1,0,25.636917,93.333333
2,59158,52,1,165,95.0,160,100,2,1,0,0,1,1,34.894399,120.000000
3,16399,55,1,164,83.0,150,100,1,1,0,0,1,1,30.859607,116.666667
4,29470,49,1,156,52.0,100,67,1,1,0,0,0,0,21.367521,78.000000
5,89915,60,1,158,83.0,150,100,1,1,0,0,1,1,33.247877,116.666667


## Feature Selection

**Dropping Redundant Columns**

In [13]:
cols_drop = ["id","height","weight","ap_hi","ap_lo"]

df = df.drop(cols_drop, axis = 1)

In [14]:
df.head()

,age,gender,cholesterol,gluc,smoke,alco,active,cardio,bmi,mabp
0,52,2,1,1,0,0,0,1,31.626276,93.333333
1,47,1,1,1,0,0,1,0,25.636917,93.333333
2,52,1,2,1,0,0,1,1,34.894399,120.000000
3,55,1,1,1,0,0,1,1,30.859607,116.666667
4,49,1,1,1,0,0,0,0,21.367521,78.000000


**Creating Feature Matrices**

In [15]:
X = df.drop("cardio", axis=1)
y= df["cardio"]

**Train and Test Split**

In [17]:
X_train, X_test, y_train, y_test = (train_test_split(X, y, random_state=42, test_size = 0.2))

## Build Model 

## Logistic Regression

In [18]:
# Building the model pipeline
lr_pipe = Pipeline([
    ("SS", StandardScaler()),
    ("LR", LogisticRegression())
])

In [19]:
# Hyperparameter grid search

params = {"LR__C":[0.1, 1, 5, 10], "LR__penalty":["l1","l2"]}

grid_lr = GridSearchCV(lr_pipe, params, cv=5, scoring="f1", n_jobs=-1)

grid_lr.fit(X_train, y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
20 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/pipeline.py", line 405, in fit
    self._final_estimator.fit(Xt, y, **fit_params_last_step)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py", line 1162, in fit
    solver = _check_solver(self.solver, self.penalty, self.dual)
 

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('SS', StandardScaler()),
                                       ('LR', LogisticRegression())]),
             n_jobs=-1,
             param_grid={'LR__C': [0.1, 1, 5, 10], 'LR__penalty': ['l1', 'l2']},
             scoring='f1')

In [20]:
print("Best params: ", grid_lr.best_params_)
print("Best Training F1 score: ", grid_lr.best_score_)
best_lr_model = grid_lr.best_estimator_

Best params:  {'LR__C': 0.1, 'LR__penalty': 'l2'}
Best Training F1 score:  0.6429935651338742


In [21]:
lr_training_report = classification_report(y_train, best_lr_model.predict(X_train))
print(lr_training_report)

              precision    recall  f1-score   support

           0       0.64      0.69      0.67     22378
           1       0.67      0.62      0.64     22422

    accuracy                           0.65     44800
   macro avg       0.66      0.65      0.65     44800
weighted avg       0.66      0.65      0.65     44800



**Logistic Regression Model Test Perfomance**

In [22]:
lr_testing_report = classification_report(y_test, best_lr_model.predict(X_test))
print(lr_testing_report)

              precision    recall  f1-score   support

           0       0.64      0.67      0.66      5639
           1       0.65      0.62      0.64      5561

    accuracy                           0.65     11200
   macro avg       0.65      0.65      0.65     11200
weighted avg       0.65      0.65      0.65     11200



## Decision Tree Classifier

In [23]:
# Building the model pipeline
dtc_pipe = Pipeline([
    ("SS", StandardScaler()),
    ("DTC", DecisionTreeClassifier())
])

In [24]:
# Hyperparameter grid search

params = {"DTC__min_samples_split": [6,7,8,10],
           "DTC__min_samples_leaf": [3,5,7,9],
          "DTC__max_features": ["sqrt", "log2", 0.4, 0.5, 0.8],
          "DTC__criterion": ["gini", "entropy"],
          "DTC__max_depth":[7, 8, 9, 10]}

grid_dtc = GridSearchCV(dtc_pipe, params, cv=5, scoring="f1", n_jobs=-1)

grid_dtc.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('SS', StandardScaler()),
                                       ('DTC', DecisionTreeClassifier())]),
             n_jobs=-1,
             param_grid={'DTC__criterion': ['gini', 'entropy'],
                         'DTC__max_depth': [7, 8, 9, 10],
                         'DTC__max_features': ['sqrt', 'log2', 0.4, 0.5, 0.8],
                         'DTC__min_samples_leaf': [3, 5, 7, 9],
                         'DTC__min_samples_split': [6, 7, 8, 10]},
             scoring='f1')

In [25]:
print("Best params: ", grid_dtc.best_params_)
print("Best Training F1 score: ", grid_dtc.best_score_)
best_dtc_model = grid_dtc.best_estimator_

Best params:  {'DTC__criterion': 'entropy', 'DTC__max_depth': 8, 'DTC__max_features': 0.8, 'DTC__min_samples_leaf': 5, 'DTC__min_samples_split': 10}
Best Training F1 score:  0.7184137797149674


In [26]:
dtc_training_report = classification_report(y_train, best_dtc_model.predict(X_train))
print(dtc_training_report)

              precision    recall  f1-score   support

           0       0.72      0.77      0.74     22378
           1       0.75      0.70      0.73     22422

    accuracy                           0.74     44800
   macro avg       0.74      0.74      0.74     44800
weighted avg       0.74      0.74      0.74     44800



**Decision Tree Model Test Perfomance**

In [27]:
dtc_testing_report = classification_report(y_test, best_dtc_model.predict(X_test))
print(dtc_testing_report)

              precision    recall  f1-score   support

           0       0.72      0.75      0.73      5639
           1       0.73      0.70      0.72      5561

    accuracy                           0.73     11200
   macro avg       0.73      0.73      0.73     11200
weighted avg       0.73      0.73      0.73     11200



In [ ]:
# precision    recall  f1-score   support

#            0       0.72      0.75      0.73      5557
#            1       0.74      0.71      0.72      5643

#     accuracy                           0.73     11200
#    macro avg       0.73      0.73      0.73     11200
# weighted avg       0.73      0.73      0.73     11200

## Random Forest Classifier

In [28]:
# Building the model pipeline
rfc_pipe = Pipeline([
    ("SS", StandardScaler()),
    ("RFC", RandomForestClassifier())
])

In [29]:
# Randomized Hyperparameter Search
params = {"RFC__n_estimators": np.arange(100,300,25) , 
          "RFC__max_depth": np.arange(4,8,2), 
          "RFC__min_samples_split":np.arange(8,15,2),
          "RFC__min_samples_leaf":np.arange(8,15,2), 
          "RFC__max_features": ["sqrt","log2",0.5, 0.8, 0.9],
          "RFC__max_samples" : np.arange(0.5,0.9,0.1)
         }

rand_rfc = RandomizedSearchCV(rfc_pipe, params, cv=4, n_iter=80, scoring="f1", n_jobs=-1)

rand_rfc.fit(X_train, y_train)

RandomizedSearchCV(cv=4,
                   estimator=Pipeline(steps=[('SS', StandardScaler()),
                                             ('RFC',
                                              RandomForestClassifier())]),
                   n_iter=80, n_jobs=-1,
                   param_distributions={'RFC__max_depth': array([4, 6]),
                                        'RFC__max_features': ['sqrt', 'log2',
                                                              0.5, 0.8, 0.9],
                                        'RFC__max_samples': array([0.5, 0.6, 0.7, 0.8]),
                                        'RFC__min_samples_leaf': array([ 8, 10, 12, 14]),
                                        'RFC__min_samples_split': array([ 8, 10, 12, 14]),
                                        'RFC__n_estimators': array([100, 125, 150, 175, 200, 225, 250, 275])},
                   scoring='f1')

In [30]:
print("Best params: ", rand_rfc.best_params_)
print("Best Training F1 score: ", rand_rfc.best_score_)
best_rand_rfc_model = rand_rfc.best_estimator_

Best params:  {'RFC__n_estimators': 200, 'RFC__min_samples_split': 14, 'RFC__min_samples_leaf': 14, 'RFC__max_samples': 0.7999999999999999, 'RFC__max_features': 0.9, 'RFC__max_depth': 6}
Best Training F1 score:  0.7193028005554843


In [31]:
rand_rfc_training_report = classification_report(y_train, best_rand_rfc_model.predict(X_train))
print(rand_rfc_training_report)

              precision    recall  f1-score   support

           0       0.72      0.77      0.74     22378
           1       0.75      0.70      0.72     22422

    accuracy                           0.73     44800
   macro avg       0.73      0.73      0.73     44800
weighted avg       0.73      0.73      0.73     44800



**Random Forest Classifier Model Test Perfomance**

In [32]:
# RandomizedSearchCV Model
rand_rfc_testing_report = classification_report(y_test, best_rand_rfc_model.predict(X_test))
print(rand_rfc_testing_report)

              precision    recall  f1-score   support

           0       0.72      0.76      0.74      5639
           1       0.74      0.70      0.72      5561

    accuracy                           0.73     11200
   macro avg       0.73      0.73      0.73     11200
weighted avg       0.73      0.73      0.73     11200



In [ ]:
# precision    recall  f1-score   support

#            0       0.72      0.77      0.74      5557
#            1       0.76      0.71      0.73      5643

#     accuracy                           0.74     11200
#    macro avg       0.74      0.74      0.74     11200
# weighted avg       0.74      0.74      0.74     11200

## The Best Model

In [33]:
reports = [lr_testing_report, dtc_testing_report, rand_rfc_testing_report]
report_names = ["Logistic Regression Test Report", "Decison Tree Classifier Test Report",
                "Random Forest Classifier Test Report"]

for report, names in zip(reports, report_names):
    print(names,"\n",report)

Logistic Regression Test Report 
               precision    recall  f1-score   support

           0       0.64      0.67      0.66      5639
           1       0.65      0.62      0.64      5561

    accuracy                           0.65     11200
   macro avg       0.65      0.65      0.65     11200
weighted avg       0.65      0.65      0.65     11200

Decison Tree Classifier Test Report 
               precision    recall  f1-score   support

           0       0.72      0.75      0.73      5639
           1       0.73      0.70      0.72      5561

    accuracy                           0.73     11200
   macro avg       0.73      0.73      0.73     11200
weighted avg       0.73      0.73      0.73     11200

Random Forest Classifier Test Report 
               precision    recall  f1-score   support

           0       0.72      0.76      0.74      5639
           1       0.74      0.70      0.72      5561

    accuracy                           0.73     11200
   macro avg     

## Saving The Best Model

In [34]:
import pickle

with open("TSEN_model.pkl", "wb") as f:
    pickle.dump(best_rand_rfc_model, f)


## Preparing Predictions For Submission

In [35]:
import pickle
with open("/kaggle/working/TSEN_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

In [46]:
def prep_data(filepath):
    df = pd.read_csv(filepath)
    # Converting the `age` column from days to years
    df["age"] = (df["age"] / 365).astype("int")
    
    # Creating nominal categories
    cols_to_change = ["gender","smoke", "alco", "active"]
    for cols in cols_to_change:
        df[cols] = pd.Categorical(df[cols], ordered=False)
        
    # Creating ordinal categories
    df["cholesterol"] = pd.Categorical(df["cholesterol"], categories = [1,2,3], ordered=True)
    df["gluc"] = pd.Categorical(df["gluc"], categories = [1,2,3], ordered=True)

    # Creating `BMI` column from `weight` and `height`
    df["bmi"] = (df["weight"] / (df["height"]/100)**2)
    
    # Creating Mean Arterial Blood Pressure `mabp` from `ap_hi` and `ap_lo`
    df["mabp"] = (df["ap_lo"] + ((1/3) * (df["ap_hi"] - df["ap_lo"])))

    cols_drop = ["id","height","weight","ap_hi","ap_lo"]
    df = df.drop(cols_drop, axis = 1)

    return df

In [44]:
def fetch_id(filepath):
    df = pd.read_csv(filepath)
    ids = df["id"].values

    return ids

In [47]:
test_ids = fetch_id("/kaggle/input/spark-2025-heart-failure-prediction/test.csv")
test_data = prep_data("/kaggle/input/spark-2025-heart-failure-prediction/test.csv")

In [49]:
test_data.head()

,age,gender,cholesterol,gluc,smoke,alco,active,bmi,mabp
0,53,1,1,1,0,0,1,24.765869,96.666667
1,57,1,1,1,0,0,1,23.046875,103.333333
2,41,2,2,1,0,0,1,28.734694,93.333333
3,51,2,1,1,0,0,1,19.789971,100.000000
4,49,1,1,1,0,0,1,29.043709,93.333333


In [52]:
test_ids[:10]

array([26681, 58585, 54339, 17273, 25420, 96794, 82860, 36971, 46401,
       35323])

In [51]:
predictions = loaded_model.predict(test_data)

predictions[:10]

array([0, 1, 0, 1, 0, 0, 0, 0, 0, 1])

In [53]:
submissions = pd.DataFrame({"id":test_ids, "cardio":predictions}).set_index("id")
print(submissions)
submissions.to_csv("submission.csv", index=True)

       cardio
id           
26681       0
58585       1
54339       0
17273       1
25420       0
...       ...
34457       1
40980       1
83726       1
11086       0
26267       1

[14000 rows x 1 columns]
